## **This notebook shows how to perform the environmental calculations and plot the results** 

In [1]:
from brightway2 import *
import bw2data as bd
import pyprind
from pathlib import Path
import pandas as pd

In [4]:
projects.set_current("marine_fuels") #<---- Project name
databaseNames = databases
myDatabaseNames = []
for databaseName in databaseNames:
    if'Circular' in databaseName:
        myDatabaseNames.append(databaseName)
myDatabaseNames.sort()

In [1]:
myDatabaseNames

NameError: name 'myDatabaseNames' is not defined

In [ ]:
GWPMethod2021 = [method for method in methods if 'IPCC 2021' in str(method) and  'climate change' 
                in str(method) and 'GWP 100a, incl. H and bio CO2' in str(method)]
method = [GWPMethod2021][0]
method

In [ ]:
import pandas as pd
import pyprind
import brightway2 as bd  # Assuming brightway2 is used for LCA calculations

# Function to perform breakdown calculations for a given activity
def breakdown_calculations(db, activity):
    activitiesList = [{activity: 1}]
    for exchange in activity.technosphere():
        activitiesList.append({bd.Database(exchange.input.key[0]).get(exchange.input.key[1]): exchange.amount})
    calculationSetup = {'inv': activitiesList, 'ia': method}
    calculation_setups['breakdown'] = calculationSetup
    myLCA = bd.MultiLCA('breakdown')
    results = pd.DataFrame(myLCA.results.transpose(), columns=[str(list(i.keys())[0]).split('\'')[1] for i in activitiesList], index=pd.MultiIndex.from_tuples(method))
    results = results.sort_index().drop(index=[i for i in results.index if i[0] == 'rest'])
    directEmissions = pd.DataFrame([0 if abs(results.iloc[r, 0] - results.iloc[r, 1:].sum()) / abs(results.iloc[r, 0]) < 1e-5 else results.iloc[r, 0] - results.iloc[r, 1:].sum() for r in range(len(results.index))],
                                   columns=['direct emissions'],
                                   index=results.index)
    results = pd.concat([results, directEmissions], axis=1)
    results['database'] = db.name  # Assuming db has a 'name' attribute
    results['location'] = activity['location']
    return results

# Initialize an empty list to store result dataframes
results_df_HFO = []
results_df_MeOH = []
results_df_LNG = []
for db_name in pyprind.prog_percent(myDatabaseNames):
    print(db_name)
    get_db = bd.Database(db_name)
    transport_act = [act for act in get_db if 'transport, freight' in act['name'] and "HFO" in act["name"]]
    transport_act1 = [act for act in get_db if 'transport, freight' in act['name'] and "capture" in act["name"]]
    transport = transport_act + transport_act1

    for act in transport:
        if "HFO" in act["name"]:
            results = breakdown_calculations(get_db, act)  # Pass get_db instead of db
            results.reset_index(drop=True, inplace=True)  # Reset index to ensure it's unique
            results_df_HFO.append(results)  # Append the results from each activity to the list
        if "Methanol" in act["name"]:
            results = breakdown_calculations(get_db, act)  # Pass get_db instead of db
            results.reset_index(drop=True, inplace=True)  # Reset index to ensure it's unique
            results_df_MeOH.append(results)  # Append the results from each activity to the list
        if "LNG" in act["name"]:
            results = breakdown_calculations(get_db, act)  # Pass get_db instead of db
            results.reset_index(drop=True, inplace=True)  # Reset index to ensure it's unique
            results_df_LNG.append(results)  # Append the results from each activity to the list

def remove_duplicate_columns(df):
    """
    Removes or renames duplicate columns in a DataFrame.
    """
    new_columns = []
    seen_columns = set()
    for col in df.columns:
        new_col = col
        i = 1
        while new_col in seen_columns:
            new_col = f"{col}_{i}"
            i += 1
        seen_columns.add(new_col)
        new_columns.append(new_col)
    df.columns = new_columns
    return df



#---------------------------------------Changing HFO dataframe
# Apply the function to each DataFrame to ensure all columns are unique
unique_col_dfs = [remove_duplicate_columns(df) for df in results_df_HFO]

# Now, find the union of all columns across the DataFrames
all_columns = set().union(*(df.columns for df in unique_col_dfs))

# Reindex each DataFrame to have the same set of columns
reindexed_dfs = [df.reindex(columns=all_columns) for df in unique_col_dfs]
df_HFO = pd.concat(reindexed_dfs, ignore_index=True)

target_columns1 = [col for col in df_HFO.columns if "market for bilge oil" in col]
target_columns2 = [col for col in df_HFO.columns if "heavy fuel oil" in col]
df_HFO['Bilge oil'] = df_HFO[target_columns1].sum(axis=1)
df_HFO['Fuel'] = df_HFO[target_columns2].sum(axis=1)

df_HFO.drop(columns=target_columns1, inplace=True)
df_HFO.drop(columns=target_columns2, inplace=True)

df_HFO.rename(columns={
    "port facilities construction": "Port", 
    "market for container ship" : "Container ship, construction",
    "market for maintenance, container ship" : "Container ship, maintenance"}, inplace=True)

new_columns = df_HFO.columns.tolist()

for i, col in enumerate(df_HFO.columns):
    if "transport" in col:
        new_columns[i] = "Total"
    elif "direct" in col:
        new_columns[i] = "Combustion emissions"

# Assign the new column names to the DataFrame
df_HFO.columns = new_columns
# Step 1: Specified order of columns
specified_order = ["Container ship, construction", "Container ship, maintenance","Bilge oil", "Port", "Fuel", "Combustion emissions", "Total", "database", "location"]

# Step 2: Identify additional columns
additional_columns = [col for col in df_HFO.columns if col not in specified_order]

# Step 3: Concatenate specified order with additional columns
new_order = specified_order + additional_columns

# Step 4: Reorder the DataFrame columns
df_HFO = df_HFO[new_order]
df_HFO["Ship"]= "HFO"


#---------------------------------------Changing Methanol dataframe
# Apply the function to each DataFrame to ensure all columns are unique
unique_col_dfs = [remove_duplicate_columns(df) for df in results_df_MeOH]

# Now, find the union of all columns across the DataFrames
all_columns = set().union(*(df.columns for df in unique_col_dfs))

# Reindex each DataFrame to have the same set of columns
reindexed_dfs = [df.reindex(columns=all_columns) for df in unique_col_dfs]
df_MeOH = pd.concat(reindexed_dfs, ignore_index=True)


target_columns = [col for col in df_MeOH.columns if "market for bilge oil" in col]
df_MeOH['Bilge oil'] = df_MeOH[target_columns].sum(axis=1)
df_MeOH.drop(columns=target_columns, inplace=True)
df_MeOH.rename(columns={
    "market for ammonia, anhydrous, liquid": "Ammonia",
    "port facilities construction": "Port", 
    "market for monoethanolamine": "Monoethanolamine", 
    "market for container ship" : "Container ship, construction",
    "market for maintenance, container ship" : "Container ship, maintenance"}, inplace=True)

new_columns = df_MeOH.columns.tolist()

for i, col in enumerate(df_MeOH.columns):
    if "transport" in col:
        new_columns[i] = "Total"
    elif "methanol" in col:
        new_columns[i] = "Fuel"
    elif "direct" in col:
        new_columns[i] = "Combustion emissions"

# Assign the new column names to the DataFrame
df_MeOH.columns = new_columns
# Step 1: Specified order of columns
specified_order = ["Container ship, construction", "Container ship, maintenance","Bilge oil", "Port", "Fuel", "Ammonia", "Monoethanolamine", "Combustion emissions", "Total", "database", "location"]

# Step 2: Identify additional columns
additional_columns = [col for col in df_MeOH.columns if col not in specified_order]

# Step 3: Concatenate specified order with additional columns
new_order = specified_order + additional_columns

# Step 4: Reorder the DataFrame columns
df_MeOH = df_MeOH[new_order]
df_MeOH["Ship"]= "Methanol"


# #---------------------------------------Changing LNG dataframe
unique_col_dfs = [remove_duplicate_columns(df) for df in results_df_LNG]
all_columns = set().union(*(df.columns for df in unique_col_dfs))
# Reindex each DataFrame to have the same set of columns
reindexed_dfs = [df.reindex(columns=all_columns) for df in unique_col_dfs]
df_LNG = pd.concat(reindexed_dfs, ignore_index=True)

target_columns = [col for col in df_LNG.columns if "market for bilge oil" in col]
df_LNG['Bilge oil'] = df_LNG[target_columns].sum(axis=1)
df_LNG.drop(columns=target_columns, inplace=True)
df_LNG.rename(columns={
    "market for monoethanolamine": "Monoethanolamine", 
    "port facilities construction": "Port", 
    "market for container ship" : "Container ship, construction",
    "market for maintenance, container ship" : "Container ship, maintenance"}, inplace=True)

new_columns = df_LNG.columns.tolist()

for i, col in enumerate(df_LNG.columns):
    if "transport" in col:
        new_columns[i] = "Total"
    elif "natural gas" in col:
        new_columns[i] = "Fuel"
    elif "direct" in col:
        new_columns[i] = "Combustion emissions"

# Assign the new column names to the DataFrame
df_LNG.columns = new_columns
# Step 1: Specified order of columns
specified_order = ["Container ship, construction", "Container ship, maintenance","Bilge oil", "Port", "Fuel", "Monoethanolamine", "Combustion emissions", "Total", "database", "location"]

# Step 2: Identify additional columns
additional_columns = [col for col in df_LNG.columns if col not in specified_order]


# Step 3: Concatenate specified order with additional columns
new_order = specified_order + additional_columns

# Step 4: Reorder the DataFrame columns
df_LNG = df_LNG[new_order]
df_LNG["Ship"]= "LNG"

dfs_concat = [df_HFO, df_MeOH, df_LNG]
final_results = pd.concat(dfs_concat, axis=0)
final_results.fillna(0, inplace=True)
target_columns = [col for col in final_results.columns if "ship" in col]
final_results['Container ship'] = final_results[target_columns].sum(axis=1)
final_results.drop(columns=target_columns, inplace=True)

new_order_order = ["Container ship","Bilge oil", "Port", "Fuel", "Monoethanolamine", "Ammonia", "Combustion emissions", "Total", "database", "Ship", "location"]

final_results = final_results[new_order_order]


final_results

Plotting

In [1]:
path = "Y:\projects\4-Capture on-board renewable ships -Valentin\manuscript\Calculations\figures"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# Assuming 'final_results' is your DataFrame
final_results['Year'] = final_results['database'].apply(lambda x: x.split("-")[-1])  # Extract year from 'database' column

# Plotting setup
fig, axs = plt.subplots(1, 2, figsize=(7, 4))  # Create subplots for each year group
plt.subplots_adjust(wspace=0.000000001)  # Adjust the width of the space between subplots
colors = ["#143f84ff", '#39627bff', '#032927ff','#19b6c2ff', '#bbbf2eff', '#31865bff', '#7787a0ff']
years = ['2030', '2050']
scatter_color = 'black'  # Color for scatter plot points

for index, (ax, year) in enumerate(zip(axs, years)):
    # Filter Data for the Current Year
    results_filtered = final_results[final_results['Year'] == year]
    # Prepare Data for Plotting
    results_filtered_for_bars = results_filtered.drop(columns=['Total', 'Year', 'database', "location", "Ship"])
    # Plotting with adjusted bar width
    bars = results_filtered_for_bars.plot(kind='bar', stacked=True, ax=ax, legend=index==1, color=colors, width=0.5)  # Adjusted width here
    ax.set_title(year, fontweight='bold', fontsize = 11, fontname="Arial",)  # Set title with bold font weight directly
    # Set y-axis to scientific notation
    ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    
    # Adding scatter plot for "Total" column
    x_ticks = bars.get_xticks()
    totals = results_filtered['Total']
    ax.scatter(x_ticks, totals, color=scatter_color, label='Total', zorder=5)
    
    # Adding total value above each scatter plot point with two significant digits
    for x, total in zip(x_ticks, totals):
        formatted_total = f'{total*10**3:.1f}'
        ax.text(x, total+0.00008, formatted_total, ha='center', va='bottom', zorder=8, fontsize=9)

    # Set x-axis labels based on the "Ship" column, changing 'LNG' to 'Natural Gas'
    ship_labels = results_filtered['Ship'].replace({'LNG': 'Natural \ngas ship', 'Methanol': 'Methanol \nship', 'HFO': 'HFO \nship'}).values
    ax.set_xticklabels(ship_labels, rotation=0, ha='center', fontname="Arial", fontsize=9)  # Adjust rotation and alignment as needed

# Adjusting spines and y-axis visibility
# Left figure (first subplot)
axs[0].spines['right'].set_visible(False)
axs[0].spines['top'].set_visible(False)

# Right figure (second subplot)
axs[1].spines['left'].set_visible(False)
axs[1].spines['right'].set_visible(False)
axs[1].spines['top'].set_visible(False)
axs[1].get_yaxis().set_visible(False)  # Hide the y-axis
# Draw a dashed horizontal line at the specified value
axs[0].axhline(y=0.00933263626796497, color='black', linestyle='--', linewidth=1,  xmin=0., xmax=0.40)
axs[0].text(x=1.18, y=0.0089, s=' HFO ship \n2025', horizontalalignment='center', color='black', fontsize=8)

# Global plot adjustments
# Apply y-axis label specifically to the left subplot
axs[0].set_ylabel(r'Global warming [kg CO$_{2}$ tkm$\mathbf{^{-1}}$]', fontsize=10, fontname="Arial", fontweight='bold')
axs[0].spines['left'].set_linewidth(1.5)
axs[0].spines['bottom'].set_linewidth(1.5)
axs[1].spines['bottom'].set_linewidth(1.5)
# Adjust the legend position for the right plot only
# axs[1].legend(loc='upper left',  bbox_to_anchor=(1, 1.1), frameon=False, fontsize=9, prop={'family': 'Arial'})
axs[1].legend(loc='upper left', bbox_to_anchor=(0.95, 0.28), frameon=False, fontsize=9, prop={'family': 'Arial'}, ncol=2, columnspacing=0.5)
# Simulate extending the bottom spine of the right subplot
# Draw a line that visually extends the bottom spine to the left
axs[1].plot([-0.2, 0], [0, 0], color="black", transform=axs[1].transAxes, clip_on=False)

plt.tight_layout()

plt.savefig(path+'environmental_assessment.svg', dpi=600, bbox_inches='tight')
plt.show()

Breakdown of fuel production

In [ ]:
import pandas as pd
import pyprind
import brightway2 as bd  # Assuming brightway2 is used for LCA calculations

# Function to perform breakdown calculations for a given activity
def breakdown_calculations(db, activity):
    activitiesList = [{activity: 1}]
    for exchange in activity.technosphere():
        activitiesList.append({bd.Database(exchange.input.key[0]).get(exchange.input.key[1]): exchange.amount})
    calculationSetup = {'inv': activitiesList, 'ia': method}
    calculation_setups['breakdown'] = calculationSetup
    myLCA = bd.MultiLCA('breakdown')
    results = pd.DataFrame(myLCA.results.transpose(), columns=[str(list(i.keys())[0]).split('\'')[1] for i in activitiesList], index=pd.MultiIndex.from_tuples(method))
    results = results.sort_index().drop(index=[i for i in results.index if i[0] == 'rest'])
    directEmissions = pd.DataFrame([0 if abs(results.iloc[r, 0] - results.iloc[r, 1:].sum()) / abs(results.iloc[r, 0]) < 1e-5 else results.iloc[r, 0] - results.iloc[r, 1:].sum() for r in range(len(results.index))],
                                   columns=['direct emissions'],
                                   index=results.index)
    results = pd.concat([results, directEmissions], axis=1)
    results['database'] = db.name  # Assuming db has a 'name' attribute
    results['location'] = activity['location']
    return results

# Initialize an empty list to store result dataframes
results_df_MeOH = []
results_df_LNG = []
for db_name in pyprind.prog_percent(myDatabaseNames):
    print(db_name)
    get_db = bd.Database(db_name)
    prod_act = [act for act in get_db if 'CO2 from ship' in act['name']]

    for act in prod_act:
        if "natural gas" in act["name"]:
            results = breakdown_calculations(get_db, act)  # Pass get_db instead of db
            results.reset_index(drop=True, inplace=True)  # Reset index to ensure it's unique
            results_df_LNG.append(results)  # Append the results from each activity to the list
        if "methanol" in act["name"]:
            results = breakdown_calculations(get_db, act)  # Pass get_db instead of db
            results.reset_index(drop=True, inplace=True)  # Reset index to ensure it's unique
            results_df_MeOH.append(results)  # Append the results from each activity to the list
        

def remove_duplicate_columns(df):
    """
    Removes or renames duplicate columns in a DataFrame.
    """
    new_columns = []
    seen_columns = set()
    for col in df.columns:
        new_col = col
        i = 1
        while new_col in seen_columns:
            new_col = f"{col}_{i}"
            i += 1
        seen_columns.add(new_col)
        new_columns.append(new_col)
    df.columns = new_columns
    return df

#---------------------------------------Changing Methanol dataframe
# Apply the function to each DataFrame to ensure all columns are unique
unique_col_dfs = [remove_duplicate_columns(df) for df in results_df_MeOH]

# Now, find the union of all columns across the DataFrames
all_columns = set().union(*(df.columns for df in unique_col_dfs))

# Reindex each DataFrame to have the same set of columns
reindexed_dfs = [df.reindex(columns=all_columns) for df in unique_col_dfs]
df_MeOH = pd.concat(reindexed_dfs, ignore_index=True)

df_MeOH.rename(columns={
    "hydrogen production, PEM electrolysis powered with wind, >3MW turbine, onshore, included PEM electrolyser construction": "Hydrogen",
    "methanol production, CO2 from ship and DAC, hydrogen from PEM electrolysis powered with wind, >3MW turbine, onshore": "Total", 
    "carbon dioxide, 1 bar, from direct air capture (DAC)": "Carbon dioxide", 
    "market for heat, from steam, in chemical industry" : "Heat",
    "market group for electricity, low voltage" : "Electricity", 
    "Cooling Water 20-25": "Cooling water"}, inplace=True)

new_columns = df_MeOH.columns.tolist()

for i, col in enumerate(df_MeOH.columns):
    if "direct" in col:
        new_columns[i] = "Direct emissions"

# Assign the new column names to the DataFrame
df_MeOH.columns = new_columns
# Step 1: Specified order of columns
specified_order = ["Hydrogen", "Carbon dioxide", "Electricity","Heat", "Cooling water", "Direct emissions", "Total", "database", "location"]

# Step 2: Identify additional columns
additional_columns = [col for col in df_MeOH.columns if col not in specified_order]

# Step 3: Concatenate specified order with additional columns
new_order = specified_order + additional_columns

# Step 4: Reorder the DataFrame columns
df_MeOH = df_MeOH[new_order]
df_MeOH["Fuel"]= "Methanol"


# #---------------------------------------Changing LNG dataframe
unique_col_dfs = [remove_duplicate_columns(df) for df in results_df_LNG]
all_columns = set().union(*(df.columns for df in unique_col_dfs))
# Reindex each DataFrame to have the same set of columns
reindexed_dfs = [df.reindex(columns=all_columns) for df in unique_col_dfs]
df_LNG = pd.concat(reindexed_dfs, ignore_index=True)

df_LNG.rename(columns={
    "hydrogen production, PEM electrolysis powered with wind, >3MW turbine, onshore, included PEM electrolyser construction": "Hydrogen",
    "natural gas production, CO2 from ships and DAC, hydrogen from PEM electrolysis powered with wind, >3MW turbine, onshore, included PEM electrolyser construction": "Total", 
    "carbon dioxide, 1 bar, from direct air capture (DAC)": "Carbon dioxide", 
    "market group for electricity, low voltage" : "Electricity", 
    "Cooling Water 20-25" : "Cooling water",
     "Liquefaction of 1kg of natural gas": "Liquefaction" }, inplace=True)

new_columns = df_LNG.columns.tolist()

for i, col in enumerate(df_LNG.columns):
    if "direct" in col:
        new_columns[i] = "Direct emissions"

# Assign the new column names to the DataFrame
df_LNG.columns = new_columns
# Step 1: Specified order of columns
specified_order = ["Hydrogen", "Carbon dioxide", "Electricity","Liquefaction", "Cooling water", "Direct emissions", "Total", "database", "location"]

# Step 2: Identify additional columns
additional_columns = [col for col in df_LNG.columns if col not in specified_order]


# Step 3: Concatenate specified order with additional columns
new_order = specified_order + additional_columns

# Step 4: Reorder the DataFrame columns
df_LNG = df_LNG[new_order]
df_LNG["Fuel"]= "LNG"

dfs_concat = [df_MeOH, df_LNG]
final_results = pd.concat(dfs_concat, axis=0)
final_results.fillna(0, inplace=True)

new_order_order = ["Hydrogen", "Carbon dioxide", "Electricity","Heat", "Liquefaction", "Cooling water", "Direct emissions", "Total", "database", "location", "Fuel"]

final_results = final_results[new_order_order]
final_results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

def extract_year_and_filter(df, year):
    df['Year'] = df['database'].apply(lambda x: x.split("-")[-1])
    return df[df['Year'] == year]

# Assuming 'final_results' is your DataFrame
final_results['Year'] = final_results['database'].apply(lambda x: x.split("-")[-1])  # Extract year from 'database' column

# Plotting setup
fig, axs = plt.subplots(1, 2, figsize=(4.5, 4.0), sharey=True)  # Create subplots for each year group with shared y-axis
plt.subplots_adjust(wspace=0.005)  # Adjust the width of the space between subplots (decrease this value to bring plots closer)
colors = ['#1cadb7ff', '#87f6f2ff','#22d5e2ff', '#efb358f7','#8979c1f7',"#143f84ff",'#7787a0ff']
years = ['2030', '2050']
scatter_color = 'black'  # Color for scatter plot points

for index, (ax, year) in enumerate(zip(axs, years)):
    # Filter Data for the Current Year
    results_filtered = extract_year_and_filter(final_results, year)
    # Prepare Data for Plotting
    results_filtered_for_bars = results_filtered.drop(columns=['Total', 'Year', 'database', "location", "Fuel"])
    # Plotting with adjusted bar width
    bars = results_filtered_for_bars.plot(kind='bar', stacked=True, ax=ax, legend=index==1, color=colors, width=0.6)  # Adjusted width here
    ax.set_title(year, fontweight='bold', fontsize=11, fontname="Arial")  # Set title with bold font weight directly
    
    # Adding scatter plot for "Total" column
    x_ticks = bars.get_xticks()
    totals = results_filtered['Total']
    ax.scatter(x_ticks, totals, color=scatter_color, label='Total', zorder=5)
    
    # Adding total value above each scatter plot point with two significant digits
    for x, total in zip(x_ticks, totals):
        formatted_total = f'{total:.2f}'
        ax.text(x, total + 0.017, formatted_total, ha='center', va='bottom', zorder=10, fontsize=9)

    # Set x-axis labels based on the "Ship" column, changing 'LNG' to 'Natural Gas'
    ship_labels = results_filtered['Fuel'].replace('LNG', 'Natural \ngas').values
    ax.set_xticklabels(ship_labels, rotation=0, ha='center', fontname="Arial", fontsize=10)  # Adjust rotation and alignment as needed

    # Draw a horizontal line at y=0
    ax.axhline(y=0, color='black', linewidth=1)

    # Set y-axis limits to include negative values
    ax.set_ylim(-0.2, 1)  # Adjust these values as needed

# Adjusting spines and y-axis visibility
# Left figure (first subplot)
axs[0].spines['right'].set_visible(False)
axs[0].spines['top'].set_visible(False)

# Right figure (second subplot)
axs[1].spines['left'].set_visible(False)
axs[1].spines['right'].set_visible(False)
axs[1].spines['top'].set_visible(False)
axs[1].get_yaxis().set_visible(False)  # Hide the y-axis

# Global plot adjustments
# Apply y-axis label specifically to the left subplot
axs[0].set_ylabel(r'Global warming [kg CO$_{2}$ kg$\mathbf{^{-1}}$]', fontsize=10, fontname="Arial", fontweight='bold')
axs[0].spines['left'].set_linewidth(1.5)
axs[0].spines['bottom'].set_linewidth(1.5)
axs[1].spines['bottom'].set_linewidth(1.5)
# Adjust the legend position for the right plot only
axs[1].legend(loc='upper left',  bbox_to_anchor=(0.85, 1.1), frameon=False, fontsize=10, prop={'family': 'Arial'})

# Simulate extending the bottom spine of the right subplot
# Draw a line that visually extends the bottom spine to the left
axs[1].plot([-0.2, 0], [0, 0], color="black", transform=axs[1].transAxes, clip_on=False)

plt.tight_layout()

plt.savefig(path+'fuel_environmental_assessment.svg', dpi=600, bbox_inches='tight')
plt.show()